In [0]:
%run ./create_secret


In [0]:
%pip install pymongo
%pip install pyyaml

dbutils.library.restartPython()

In [0]:
import datetime
import json
import yaml
import os
import time 
import bson
from pymongo import MongoClient
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType
 
CONFIG_PATH = "../config/collections.yaml"
mongo_uri = dbutils.secrets.get(scope="conn-db", key="cnn-mongodb-sampleflix")
client = MongoClient(mongo_uri)

database = client["sample_mflix"]
collection = database["movies"]

with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

In [0]:
def load_configs(config_path):
    if not config_path:
        print("Caminho do YAML não definido.")
    with open(CONFIG_PATH, 'r') as f:
        config = yaml.safe_load(f)
    return config
config = load_configs(CONFIG_PATH)

def load_collections(config):
    for item in config['collections']:
        print(item["collection"])
        print(item["modo_carga"])
        print(item["campo_watermark"])
        print(item["destino"])
    #return [c for c in config["collections"]]

collections = load_collections(config)
print(collections[0])

In [0]:
print(config.get('collections')[0])

In [0]:
collections = database.list_collection_names()

print(collections)

In [0]:
def extract(spark, mongo_uri, database, collection):
    return (
        spark.read
        .format("mongodb")
        .option("connection.uri", mongo_uri)
        .option("database", database)
        .option("collection", collection)
        .load()
    )


def load(df, destino):
    (
        df.write
        .format("delta")
        .mode("append")
        .saveAsTable(destino)
    )


def run_ingestion(
    spark,
    mongo_uri,
    database,
    collection,
    modo_carga,
    campo_watermark,
    destino
):
    if modo_carga not in ("full", "incremental"):
        raise ValueError(
            "modo_carga deve ser 'full' ou 'incremental'"
        )

    print(f"Iniciando ingestão: {database}.{collection}")
    print(f"Modo: {modo_carga}")

    df = extract(
        spark=spark,
        mongo_uri=mongo_uri,
        database=database,
        collection=collection
    )

    load(
        df=df,
        destino=destino
    )

    print(f"Ingestão concluída: {destino}")